# Matrix Multiplication

This notebook defines a custom function to multiply two matrices.
It uses `np.vsplit` to decompose the first matrix into rows, a `dot_product`
sub-function to compute each element, and `np.vstack` to assemble the result
row by row. The result is verified against NumPy's built-in `matmul`.

In [7]:
import numpy as np

## Helper: Dot Product

Computes the dot product of two 1-D vectors without NumPy.

In [8]:
def dot_product(v1, v2):
    if len(v1) != len(v2):
        raise ValueError('Vectors must have the same length.')

    total = 0
    for i in range(len(v1)):
        total += v1[i] * v2[i]

    return total

## Custom Matrix Multiplication Function

**Strategy:**
1. Use `np.vsplit(A, m)` to split matrix A into `m` individual row vectors of shape `(1, k)`.
2. Flatten each row of A into a 1-D vector so it can be used in a dot product.
3. Use `np.hsplit(B, n)` to split matrix B into `n` individual column vectors of shape `(k, 1)`.
4. Flatten each column of B into a 1-D vector so it can be used in a dot product.
5. For each row of A, compute its dot product with every column of B to produce one result row.
6. Assemble the result rows into the final `(m, n)` matrix with `np.vstack`.

In [9]:
def multiply_matrix(A, B):
    A = np.array(A, dtype=float)
    B = np.array(B, dtype=float)

    m, k = A.shape
    k2, n = B.shape

    if k != k2:
        raise ValueError(
            f'Matrix dimensions do not match: ({m}, {k}) x ({k2}, {n}).'
        )

    # Split A into m individual row arrays, then flatten each into shape (k,)
    a_rows = []
    for row in np.vsplit(A, m):
        a_rows.append(row.flatten())  # Convert the 2-D row array into a 1-D vector.


    # Split B into n individual column arrays, then flatten each into shape (k,)
    b_cols = []
    for col in np.hsplit(B, n):
        b_cols.append(col.flatten())  # Convert the 2-D column array into a 1-D vector.


    # Build the result row by row
    rows = []
    for a_row in a_rows:
        elements = []
        for b_col in b_cols:
            elements.append(dot_product(a_row, b_col))
        rows.append(np.array(elements))

    # vstack the row vectors to form the (m, n) result matrix
    return np.vstack(rows)


## Verify Using NumPy

In [10]:
import time

A = np.array([[1, 2, 3],
              [4, 5, 6]], dtype=float)   # (2, 3)

B = np.array([[7,  8],
              [9,  10],
              [11, 12]], dtype=float)    # (3, 2)

res_1 = multiply_matrix(A, B)
res_2 = np.matmul(A, B)

# Benchmark both approaches over many runs for more stable timing
runs = 100000

time_1 = time.perf_counter()
for _ in range(runs):
    multiply_matrix(A, B)
time_1 = time.perf_counter() - time_1

time_2 = time.perf_counter()
for _ in range(runs):
    np.matmul(A, B)
time_2 = time.perf_counter() - time_2

print('Matrix A:')
print(A)
print()
print('Matrix B:')
print(B)
print()
print('Custom A @ B:')
print(res_1)
print()
print('NumPy A @ B:')
print(res_2)
print(f'Match: {np.allclose(res_1, res_2)}')
print()
print(f'Custom function time over {runs} runs: {time_1:.6f} s')
print(f'NumPy matmul time over {runs} runs:   {time_2:.6f} s')

Matrix A:
[[1. 2. 3.]
 [4. 5. 6.]]

Matrix B:
[[ 7.  8.]
 [ 9. 10.]
 [11. 12.]]

Custom A @ B:
[[ 58.  64.]
 [139. 154.]]

NumPy A @ B:
[[ 58.  64.]
 [139. 154.]]
Match: True

Custom function time over 100000 runs: 2.262178 s
NumPy matmul time over 100000 runs:   0.098613 s
